In [30]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [47]:
import re
from pathlib import Path
from collections import Counter
from typing import Dict, List 

from openpyxl import load_workbook


In [48]:
TTP_KEYWORDS = [
    r"\bphish\w*\b", r"\bphishing\b", r"\bmalspam\b", r"\bvishing\b",
    r"\brdp\b", r"\bbrute\s+force\b", r"\bpassword\s+spray\b",
    r"\bexploit\w*\b", r"\bvulnerab\w*\b", r"\b0-day\b",
    r"\bcobalt\s+strike\b", r"\bbeacon\b", r"\bmimikatz\b",
    r"\bpsexec\b", r"\bwmi\b", r"\bwinrm\b",
    r"\brclone\b", r"\bmeganz\b", r"\bwinscp\b",
    r"\bexfil\w*\b", r"\bdata\s+theft\b", r"\bdouble\s+extortion\b", r"\bleak\s+site\b",
    r"\bshadow\s+copy\b", r"\bvss\b", r"\bransom\s+note\b", r"\bencrypt\w*\b",
    r"\bscheduled\s+task\b", r"\bgpo\b", r"\bgroup\s+policy\b",
]

DFIR_CASE_STUDY_SIGNALS = [
    r"\bcase\s+summary\b", r"\bintrusion\b", r"\bwe\s+observed\b", r"\btimeline\b",
    r"\bbeach\s*head\b", r"\bday\s+\d+\b", r"\bincident\s+response\b", r"\bforensic\w*\b",
]

PLATFORM_PATTERNS = {
    "ESXi": [r"\besxi\b", r"\bvmware\b", r"\bvcenter\b"],
    "Linux": [r"\blinux\b", r"\bubuntu\b", r"\bdebian\b", r"\bcentos\b", r"\brhel\b", r"\bred\s*hat\b"],
    "Windows": [r"\bwindows\b", r"\bactive\s+directory\b", r"\bdomain\s+controller\b", r"\blsass\b"],
}

DEPTH_HIGH = [
    r"\batt&ck\b", r"\bmitre\b", r"\btelemetry\b", r"\bioc(s)?\b", r"\byara\b", r"\bsigma\b",
    r"\bpcap\b", r"\bmemory\s+dump\b", r"\breverse\s+engineering\b", r"\bprocess\s+injection\b",
    r"\blog(s)?\b", r"\bevidence\b", r"\bdetection\s+rule\b",
]
DEPTH_MED = [
    r"\bkill\s+chain\b", r"\blateral\s+movement\b", r"\bprivilege\s+escalation\b",
    r"\bexfiltration\b", r"\bpersistence\b", r"\bdiscovery\b",
]

RED_FLAG_PATTERNS = {
    "Marketing": [
        r"\b(contact\s+us|pricing|book\s+a\s+demo|request\s+a\s+demo)\b",
        r"\bour\s+product\b", r"\bwebinar\b", r"\bsponsored\b",
    ],
    "News": [
        r"\bbreaking\s+news\b", r"\bheadlines\b", r"\bnews\s+update\b",
    ],
    "LowSignal": [
        r"\b(what\s+is\s+ransomware|ransomware\s+explained)\b"
    ]
}

def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower())

def any_hit(text: str, patterns: List[str]) -> bool:
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)

def count_hits(text: str, patterns: List[str]) -> int:
    return sum(len(re.findall(p, text, flags=re.IGNORECASE)) for p in patterns)

def detect_platform(text: str) -> str:
    hits = {plat: count_hits(text, pats) for plat, pats in PLATFORM_PATTERNS.items()}
    chosen = [p for p, h in hits.items() if h > 0]
    if not chosen:
        return ""   # keep empty so rubric score adds 0
    if len(chosen) == 1:
        return chosen[0]
    return "Mixed"

def detect_depth(text: str) -> str:
    high = count_hits(text, DEPTH_HIGH)
    med = count_hits(text, DEPTH_MED)
    if high >= 6:
        return "High"
    if high >= 2 or med >= 4:
        return "Medium"
    return "Low"

def detect_red_flags(text: str) -> str:
    flags = [label for label, pats in RED_FLAG_PATTERNS.items() if any_hit(text, pats)]
    return "/".join(flags)


In [49]:
def load_family_aliases_xlsx(family_list_path: str) -> Dict[str, str]:
    wb = load_workbook(family_list_path)
    ws = wb.active
    headers = {}
    for c in range(1, ws.max_column + 1):
        v = ws.cell(row=1, column=c).value
        if isinstance(v, str) and v.strip():
            headers[v.strip()] = c

    fam_col = headers.get("Ransomware_Family_Name")
    alias_col = headers.get("Alias_Names")
    if not fam_col:
        raise ValueError("Could not find 'Ransomware_Family_Name' column in family list.")

    alias_map = {}
    for r in range(2, ws.max_row + 1):
        fam = ws.cell(row=r, column=fam_col).value
        if not fam:
            continue
        fam = str(fam).strip()
        alias_map[fam.lower()] = fam

        if alias_col:
            aliases = ws.cell(row=r, column=alias_col).value
            if aliases and str(aliases).strip() and str(aliases).strip() != "—":
                for a in [x.strip() for x in str(aliases).split(",")]:
                    if a:
                        alias_map[a.lower()] = fam
    return alias_map

def detect_specific_family(text: str, alias_map: Dict[str, str]) -> bool:
    for alias_lower in alias_map.keys():
        alias_re = re.escape(alias_lower).replace(r"\ ", r"\s+")
        if re.search(rf"\b{alias_re}\b", text, flags=re.IGNORECASE):
            return True
    return False


In [50]:
alias_map = load_family_aliases_xlsx("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/rubrics/Ransomware_Family_Coverage_List.xlsx")


In [52]:
out_path = "/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/rubrics/Video_Selection_Rubric_Rebuilt_WITH_FORMULAS.xlsx"

rubric_df.to_excel(out_path,index=False)

print("Saved:",out_path)

Saved: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/rubrics/Video_Selection_Rubric_Rebuilt_WITH_FORMULAS.xlsx


In [53]:
# If transcripts are in the notebook directory, use "."
# If they're in a folder, use "transcripts"

alias_map = None
# Uncomment if you have the family list file:
alias_map = load_family_aliases_xlsx("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/rubrics/Ransomware_Family_Coverage_List.xlsx")

fill_rubric(
    rubric_xlsx="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/rubrics/Video_Selection_Rubric_Rebuilt_WITH_FORMULAS.xlsx",
    transcripts_dir="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts",  # or "transcripts"
    out_xlsx="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Video_Selection_Rubric_AutoScore_Filled.xlsx",
    alias_map=alias_map
)


Saved filled rubric to: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Video_Selection_Rubric_AutoScore_Filled.xlsx


In [46]:
fill_rubric(
    rubric_xlsx="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/rubrics/Video_Selection_Rubric_Rebuilt_WITH_FORMULAS.xlsx",
    transcripts_dir="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts",
    out_xlsx="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Video_Selection_Rubric_AutoScore_Filled.xlsx",
    alias_map=alias_map
)

Saved filled rubric to: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Video_Selection_Rubric_AutoScore_Filled.xlsx


In [25]:
import pandas as pd
import json
from pathlib import Path

BASE = Path("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent")

template_path = BASE / "rubrics/Video_Selection_Rubric_AutoScore.xlsx"
transcripts_root = BASE / "transcripts"

template = pd.read_excel(template_path)

rows = []

for txt in transcripts_root.rglob("*.txt"):

    vid = txt.stem
    meta_path = txt.with_suffix(".meta.json")

    meta = {}
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())

    row = {col:"" for col in template.columns}

    row["Video_ID"] = vid
    row["Channel_ID"] = meta.get("Channel_ID","")
    row["Channel_Name"] = meta.get("Channel_Name","")
    row["Video_Title"] = meta.get("Video_Title","")
    row["YouTube_Video_ID"] = meta.get("YouTube_Video_ID","")
    row["PublishedAt"] = meta.get("PublishedAt","")

    duration = meta.get("DurationSeconds")
    if duration:
        row["Video_Length_Minutes"] = round(duration/60,2)

    row["Transcript_Available (Yes/No)"] = "Yes"

    rows.append(row)

rubric_df = pd.DataFrame(rows)

print("Videos discovered:", len(rubric_df))

Videos discovered: 440


In [26]:
rebuilt_path = BASE / "rubrics/Video_Selection_Rubric_Rebuilt.xlsx"

rubric_df.to_excel(rebuilt_path,index=False)

print("Saved:", rebuilt_path)

Saved: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/rubrics/Video_Selection_Rubric_Rebuilt.xlsx


In [27]:
def fill_rubric(
    rubric_xlsx: str,
    transcripts_dir: str,
    out_xlsx: str,
    alias_map: Dict[str, str] | None = None,
):
    wb = load_workbook(rubric_xlsx)
    ws = wb["Video_Selection_Rubric"] if "Video_Selection_Rubric" in wb.sheetnames else wb.active

    # Map headers
    headers = {}
    for c in range(1, ws.max_column + 1):
        v = ws.cell(row=1, column=c).value
        if isinstance(v, str) and v.strip():
            headers[v.strip()] = c

    def h(name: str) -> int:
        if name not in headers:
            raise KeyError(f"Missing column in rubric: {name}")
        return headers[name]

    COL_VID = h("Video_ID")
    COL_FAM_MENTION = h("Ransomware_Family_Mentioned (Yes/No)")
    COL_SPEC_FAM = h("Specific_Family_Named (Yes/No)")
    COL_TTPS = h("TTPs_Mentioned (Yes/No)")
    COL_DFIR = h("DFIR_Case_Study (Yes/No)")
    COL_PLATFORM = h("Platform_Mentioned (Windows/Linux/ESXi)")
    COL_TRANSCRIPT = h("Transcript_Available (Yes/No)")
    COL_DEPTH = h("Estimated_Technical_Depth (Low/Medium/High)")
    COL_REDFLAG = h("Red_Flags (News/Marketing/LowSignal)")

    tdir = Path(transcripts_dir)

    def set_if_empty(r: int, c: int, value: str):
        cur = ws.cell(row=r, column=c).value
        if cur is None or str(cur).strip() == "":
            ws.cell(row=r, column=c, value=value)

    # Data rows start at row 3 (row 2 is guidance)
    for r in range(3, ws.max_row + 1):
        vid = ws.cell(row=r, column=COL_VID).value
        if not vid:
            continue
        vid = str(vid).strip()
        if not vid or vid.lower().startswith("scoring guidance"):
            continue

        tfile = tdir / f"{vid}.txt"
        if not tfile.exists():
            set_if_empty(r, COL_TRANSCRIPT, "No")
            continue

        text = normalize(tfile.read_text(encoding="utf-8", errors="ignore"))
        set_if_empty(r, COL_TRANSCRIPT, "Yes")

        # Family mentions
        has_specific = False
        if alias_map:
            has_specific = detect_specific_family(text, alias_map)

        if has_specific:
            set_if_empty(r, COL_SPEC_FAM, "Yes")
            set_if_empty(r, COL_FAM_MENTION, "Yes")
        else:
            has_ransomware_word = bool(re.search(r"\bransomware\b", text, flags=re.IGNORECASE))
            set_if_empty(r, COL_FAM_MENTION, "Yes" if has_ransomware_word else "No")
            set_if_empty(r, COL_SPEC_FAM, "No")

        # TTPs / DFIR / Platform / Depth / Red flags
        set_if_empty(r, COL_TTPS, "Yes" if any_hit(text, TTP_KEYWORDS) else "No")
        set_if_empty(r, COL_DFIR, "Yes" if any_hit(text, DFIR_CASE_STUDY_SIGNALS) else "No")

        platform = detect_platform(text)
        if platform:
            set_if_empty(r, COL_PLATFORM, platform)

        set_if_empty(r, COL_DEPTH, detect_depth(text))

        flags = detect_red_flags(text)
        if flags:
            set_if_empty(r, COL_REDFLAG, flags)

    wb.save(out_xlsx)
    print(f"Saved filled rubric to: {out_xlsx}")


In [28]:
fill_rubric(
    rubric_xlsx="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/rubrics/Video_Selection_Rubric_Rebuilt.xlsx",
    transcripts_dir="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts",
    out_xlsx="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Video_Selection_Rubric_AutoScore_Filled.xlsx",
    alias_map=alias_map
)

Saved filled rubric to: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Video_Selection_Rubric_AutoScore_Filled.xlsx


In [29]:
# Load the filled Rubric
import pandas as pd
from pathlib import Path

BASE = Path("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent")

rubric_path = BASE / "outputs/Video_Selection_Rubric_AutoScore_Filled.xlsx"

df = pd.read_excel(rubric_path)

print("Total rubric rows:", len(df))
df.head()

Total rubric rows: 440


,Video_ID,Channel_ID,Channel_Name,Video_Title,Ransomware_Family_Mentioned (Yes/No),Specific_Family_Named (Yes/No),TTPs_Mentioned (Yes/No),DFIR_Case_Study (Yes/No),Platform_Mentioned (Windows/Linux/ESXi),Transcript_Available (Yes/No),Video_Length_Minutes,Estimated_Technical_Depth (Low/Medium/High),Recency (Pre-2020 / 2020-2022 / 2023+),Red_Flags (News/Marketing/LowSignal),Selection_Score,Include (Yes/No),Notes,YouTube_Video_ID,PublishedAt
0,V0210,C22,Threatpost,Anti-Encryption Bills Put Tech Firms Under ‘Fu...,NaN,NaN,NaN,NaN,NaN,Yes,19.25,NaN,NaN,NaN,NaN,NaN,NaN,lFS959M-8NU,2020-07-27T14:36:12Z
1,V0205,C22,Threatpost,Patrick Wardle: Hackers Abusing Apple Technolo...,NaN,NaN,NaN,NaN,NaN,Yes,20.65,NaN,NaN,NaN,NaN,NaN,NaN,gtLXRQFCgOg,2020-12-21T20:32:36Z
2,V0212,C22,Threatpost,Cloud Deployment Mistakes Leading to Breaches,NaN,NaN,NaN,NaN,NaN,Yes,8.05,NaN,NaN,NaN,NaN,NaN,NaN,2sZ6x5GrdCY,2020-03-18T20:00:56Z
3,V0203,C22,Threatpost,Malware Partnerships: A Formidable Threat to B...,NaN,NaN,NaN,NaN,NaN,Yes,13.32,NaN,NaN,NaN,NaN,NaN,NaN,H7b704TFhW4,2021-02-26T14:57:53Z
4,V0204,C22,Threatpost,"Email Security Attacks End in Financial Ruin, ...",NaN,NaN,NaN,NaN,NaN,Yes,25.80,NaN,NaN,NaN,NaN,NaN,NaN,VQO8HQSA54I,2021-02-11T15:30:10Z


In [8]:
# keep only Include = Yes
selected = df[df["Include (Yes/No)"].str.lower() == "yes"].copy()

print("Selected videos:", len(selected))
selected.head()

Selected videos: 0


,Video_ID,Channel_ID,Channel_Name,Video_Title,Ransomware_Family_Mentioned (Yes/No),Specific_Family_Named (Yes/No),TTPs_Mentioned (Yes/No),DFIR_Case_Study (Yes/No),Platform_Mentioned (Windows/Linux/ESXi),Transcript_Available (Yes/No),Video_Length_Minutes,Estimated_Technical_Depth (Low/Medium/High),Recency (Pre-2020 / 2020-2022 / 2023+),Red_Flags (News/Marketing/LowSignal),Selection_Score,Include (Yes/No),Notes
